## Análise do Agronegócio Brasileiro — Commodities e Produção Agrícola

**Objetivo:** Coletar, tratar e consolidar dados de preços de commodities (soja, milho e dólar) e produção agrícola para análise exploratória.

**Fontes:**
- Preços: [CEPEA/ESALQ](https://cepea.esalq.usp.br)
- Produção: [IBGE - SIDRA](https://sidra.ibge.gov.br)

---
#### 1. Tratamento dos dados das commodities
##### - Soja / Milho / Dólar

In [186]:
# Importando bibliotecas 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

In [187]:
#Caminhos
path_dolar = r"C:\Users\lucas.takabayashi\Desktop\GitHub\agro-analysis\data\raw\dolar.xlsx"
path_milho = r"C:\Users\lucas.takabayashi\Desktop\GitHub\agro-analysis\data\raw\milho.xlsx"
path_soja = r"C:\Users\lucas.takabayashi\Desktop\GitHub\agro-analysis\data\raw\soja.xlsx"

In [188]:
#Criação de um DataFrame de data com início de 01/01/2020 a 30/04/2026
df_data = pd.DataFrame({
        'data': pd.date_range(start='2020-01-01', end='2026-04-30', freq='C', )
    })

In [189]:
#Adicionando colunas de Ano, mês e ano mês
df_data['ano'] = df_data['data'].dt.year
df_data['mes'] = df_data['data'].dt.month
df_data['ano_mes'] = df_data['data'].dt.to_period('M')


In [190]:
#Criando função para ler dados e padronizar colunas
def tratamento_dados(caminho, header=1):

    #Lendo arquivo e padronizando colunas e casas decimais
    df = pd.read_excel(caminho, header=header, decimal=',')

    #Padronizando colunas
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')

    #Convertendo data
    df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')

    return df

In [191]:
#Aplicando função em cada dataset
df_dolar = tratamento_dados(path_dolar,header=3)
df_milho = tratamento_dados(path_milho,header=3)
df_soja = tratamento_dados(path_soja,header=3)

In [192]:
print(
    f'df_dolar colunas: {list(df_dolar.columns)} \n'
    f'df_soja  colunas: {list(df_soja.columns)} \n'
    f'df_milho colunas: {list(df_milho.columns)} \n'
)

df_dolar colunas: ['data', 'à_vista_r$'] 
df_soja  colunas: ['data', 'à_vista_r$', 'à_vista_us$'] 
df_milho colunas: ['data', 'à_vista_r$', 'à_vista_us$'] 



In [193]:
#Renomeando as colunas principais
df_dolar.rename(columns={'à_vista_r$': 'dolar'}, inplace=True)
df_soja.rename(columns={'à_vista_r$': 'soja_scs'}, inplace=True)
df_milho.rename(columns={'à_vista_r$': 'milho_scs'}, inplace=True)

In [194]:
#Juntando os dataframes em um único só
df_main = (df_data.merge(df_dolar, how='left', on='data')
    .merge(df_soja, how='left', on='data')
    .merge(df_milho, how='left', on='data'))

In [195]:
# Verificando colunas após o merge
df_main.columns

Index(['data', 'ano', 'mes', 'ano_mes', 'dolar', 'soja_scs', 'à_vista_us$_x',
       'milho_scs', 'à_vista_us$_y'],
      dtype='object')

In [196]:
#Dropando colunas de dólar
df_main.drop(columns=['à_vista_us$_x','à_vista_us$_y'], inplace=True)

In [197]:
#Preenchendo valores vazios com o último valor
df_main.ffill(inplace=True)

In [198]:
#Dropando os valores vazios
df_main.dropna(inplace=True)

In [199]:
#Criando colunas de variação de preço por dia
cols = ['soja_scs','milho_scs','dolar']
for i in cols:
    df_main[f'{i}_var'] = (df_main[i].pct_change() * 100).round(2)

In [201]:
#Gravando o novo df unificado e tratado
df_main.to_excel(r'C:\Users\lucas.takabayashi\Desktop\GitHub\agro-analysis\data\clean\commodities.xlsx', index=False)

#### 2. Tratamento de dados da produção agrícola

In [158]:
#Caminhos
path_producao_soja = r'C:\Users\lucas.takabayashi\Desktop\GitHub\agro-analysis\data\raw\producao_soja.xlsx'
path_producao_milho = r'C:\Users\lucas.takabayashi\Desktop\GitHub\agro-analysis\data\raw\producao_milho.xlsx'

In [160]:
#Importando datasets
df_producao_soja = pd.read_excel(path_producao_soja, header=3)
df_producao_milho = pd.read_excel(path_producao_milho, header=3)

In [178]:
#Padronizando e renomeando colunas 
colunas_producao = {
    'Unnamed: 0': 'estado',
    'Unnamed: 1': 'ano_mes',
    'Área plantada (Hectares)': 'area_plantada_hect',
    'Área colhida (Hectares)': 'area_colhida_hect',
    'Produção (Toneladas)': 'producao_toneladas',
    'Rendimento médio (Quilogramas por Hectare)': 'rendimento_kg_por_hect'
}

for df in [df_producao_soja, df_producao_milho]:
    df.rename(columns=colunas_producao, inplace=True)

In [179]:
#Removendo a última linha do df
df_producao_soja = df_producao_soja.iloc[:-1]
df_producao_milho = df_producao_milho.iloc[:-1]

In [180]:
#Criação da coluna 'commodity' com a descrição para cada df 
df_producao_soja['commodity'] = 'soja'
df_producao_milho['commodity'] = 'milho'

C:\Users\lucas.takabayashi\AppData\Local\Temp\ipykernel_7268\3444164661.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_producao_milho['commodity'] = 'milho'


In [181]:
#Juntando os 2 df em um só
df_producao = pd.concat([df_producao_soja, df_producao_milho], axis=0)

In [182]:
#Preenchendo valores vazios com o último valor
df_producao['estado'] = df_producao['estado'].ffill()

In [183]:
#Removendo os acentos da coluna 'estado'
df_producao['estado'] = df_producao['estado'].str.replace('ã','a').str.replace('á','a')

In [184]:
#Transformando coluna 'ano_mes' no tipo date
meses_pt = {
        'janeiro': '01',
        'fevereiro': '02',
        'março': '03', 
        'abril': '04',
        'maio': '05', 
        'junho': '06', 
        'julho': '07', 
        'agosto': '08',
        'setembro': '09', 
        'outubro': '10', 
        'novembro': '11', 
        'dezembro': '12'
    }

df_producao['data'] = df_producao['ano_mes'].replace(meses_pt, regex=True)
df_producao['data'] = pd.to_datetime(df_producao['data'], format='%m %Y').dt.to_period('M')


In [185]:
#Salvando o tratado e unificado
df_producao.to_excel(r'C:\Users\lucas.takabayashi\Desktop\GitHub\agro-analysis\data\clean\producao_agricola.xlsx', index=False)